In [0]:

from pyspark.sql.functions import (
    col, sum as _sum, count as _count, when, current_timestamp
)
from pyspark.sql.types import DecimalType

CATALOG_NAME = "banking_lakehouse_db2"
SCHEMA_NAME = "gold"
TABLE_NAME = "banking_kpi_summary"
FULL_TABLE_NAME = f"`{CATALOG_NAME}`.{SCHEMA_NAME}.{TABLE_NAME}"
GOLD_PATH = f"abfss://gold@bankingdelakevishal.dfs.core.windows.net/{TABLE_NAME}/"



In [0]:
# ============================================
# GOLD: BANKING KPI SUMMARY
# ============================================

# 1. Read Silver Tables
dim_customer = spark.table(f"`{CATALOG_NAME}`.silver.customer")
dim_account = spark.table(f"`{CATALOG_NAME}`.silver.account")
dim_loan = spark.table(f"`{CATALOG_NAME}`.silver.loan")
dim_fd = spark.table(f"`{CATALOG_NAME}`.silver.fixed_deposit")
fact_tx = spark.table(f"`{CATALOG_NAME}`.silver.transaction")

# 2. Extract Individual KPI Metrics
total_cust = dim_customer.filter(col("customer_status") == "ACTIVE").count()
total_acc = dim_account.filter(col("account_status") == "ACTIVE").count()

deposits = dim_account.agg(_sum("balance")).collect()[0][0] or 0.00
fds = dim_fd.filter(col("fd_status") == "ACTIVE").agg(_sum("deposit_amount")).collect()[0][0] or 0.00
loans = dim_loan.filter(col("loan_status") == "ACTIVE").agg(_sum("principal")).collect()[0][0] or 0.00

tx_stats = fact_tx.agg(
    _count("transaction_id").alias("tx_count"),
    _sum("amount").alias("tx_vol"),
    _count(when(col("fraud_flag") == True, True)).alias("fraud_count")
).collect()[0]

# 3. Create High-Level Single-Row Dataframe
kpi_data = [(
    total_cust,
    total_acc,
    float(deposits),
    float(fds),
    float(loans),
    float(deposits + fds),  # Total Assets Under Management
    tx_stats["tx_count"],
    float(tx_stats["tx_vol"] or 0.00),
    tx_stats["fraud_count"]
)]

kpi_schema = [
    "active_customers", "active_accounts", "total_account_deposits", 
    "total_fd_deposits", "total_outstanding_loans", "total_aum", 
    "total_transactions", "total_transaction_volume", "total_fraud_transactions"
]

kpi_summary_df = (
    spark.createDataFrame(kpi_data, kpi_schema)
    .withColumn("total_account_deposits", col("total_account_deposits").cast(DecimalType(18, 2)))
    .withColumn("total_fd_deposits", col("total_fd_deposits").cast(DecimalType(18, 2)))
    .withColumn("total_outstanding_loans", col("total_outstanding_loans").cast(DecimalType(18, 2)))
    .withColumn("total_aum", col("total_aum").cast(DecimalType(18, 2)))
    .withColumn("total_transaction_volume", col("total_transaction_volume").cast(DecimalType(18, 2)))
    .withColumn("gold_ingestion_timestamp", current_timestamp())
)

# 4. Write to ADLS & Unity Catalog
kpi_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(GOLD_PATH)
kpi_summary_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(FULL_TABLE_NAME)

print(f"Banking KPI Summary Gold Table Created successfully in '{FULL_TABLE_NAME}'.")

Banking KPI Summary Gold Table Created successfully in '`banking_lakehouse_db2`.gold.banking_kpi_summary'.
